# Burmese Polarization / Harmful Language Classification — BiLSTM

Multi-label text classification for Burmese text. Each row contains `text` plus six binary labels:
`stereotype`, `vilification`, `dehumanization`, `extreme_language`, `lack_of_empathy`, and `invalidation`.

**Dataset format:** UTF-8 pipe-separated (`|`) file with a header row. Save your full dataset as `../dataset/burmese_polarization_multilabel.txt` or update `DATA_PATH` below. The text column may contain spaces; the pipe is the delimiter.

This notebook uses whitespace tokenization, which matches the syllable/word-separated style in the supplied examples. It does not perform Burmese word segmentation.

In [1]:
import pandas as pd
import numpy as np
import csv
import torch
import torch.nn as nn

from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 1. Load the dataset

In [2]:
LABELS = [
    "stereotype",
    "vilification",
    "dehumanization",
    "extreme_language",
    "lack_of_empathy",
    "invalidation",
]

DATA_PATH = "../dataset/task3_polarization_manifestation_no_id.txt"

rows = []
bad_rows = 0

with open(DATA_PATH, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f, delimiter="|", quotechar='"')
    header = next(reader, None)

    expected_header = ["text"] + LABELS
    if header is None:
        raise ValueError("The dataset file is empty.")
    if [h.strip() for h in header] != expected_header:
        raise ValueError(
            f"Unexpected header: {header}\nExpected: {expected_header}"
        )

    for line_number, row in enumerate(reader, start=2):
        if len(row) != 1 + len(LABELS):
            bad_rows += 1
            continue
        text = row[0].strip()
        try:
            labels = [int(value.strip()) for value in row[1:]]
        except ValueError:
            bad_rows += 1
            continue
        if not text or not all(value in (0, 1) for value in labels):
            bad_rows += 1
            continue
        rows.append([text] + labels)

df = pd.DataFrame(rows, columns=["text"] + LABELS)
print("Dataset shape:", df.shape)
print("Skipped malformed/invalid rows:", bad_rows)
display(df.head())
print("\nPositive examples per label:")
print(df[LABELS].sum().sort_values(ascending=False))
print("\nLabel prevalence:")
print((df[LABELS].mean() * 100).round(2).astype(str) + "%")

Dataset shape: (9632, 7)
Skipped malformed/invalid rows: 14


,text,stereotype,vilification,dehumanization,extreme_language,lack_of_empathy,invalidation
0,ကိုယ့်နိုင်ငံကြီး အေးချမ်းသွားရင် ကိုယ့်နိုင်င...,1,0,0,0,1,0
1,တကယ် လေ နိုင်ငံခြား သား တွေ အကုန် လုံး ကို တစ်...,1,0,0,0,1,0
2,ဂျပန် ထဲ မြန်မာ တွေ ဝင် လာ ပြီ ဆိုတော့ သူတို့ ...,1,0,0,0,0,0
3,ဒီ အကြောင်း တွေ ကို ဂျပန် လို ပြော ပြီး တင် ...,1,0,0,0,0,0
4,တရုတ် တွေ က ဂျပန် ကို မုန်း တယ် ပြော ပြီး ဘာလိ...,1,0,0,0,0,0



Positive examples per label:
vilification        3233
extreme_language    2015
invalidation        1592
stereotype          1515
lack_of_empathy     1151
dehumanization      1046
dtype: int64

Label prevalence:
stereotype          15.73%
vilification        33.57%
dehumanization      10.86%
extreme_language    20.92%
lack_of_empathy     11.95%
invalidation        16.53%
dtype: object


## 2. Clean and validate labels

In [3]:
print("Rows before cleaning:", len(df))
df = df.dropna(subset=["text"] + LABELS).copy()
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].ne("")].copy()
df[LABELS] = df[LABELS].astype("int64")

assert df[LABELS].isin([0, 1]).all().all()
assert df[LABELS].notna().all().all()
print("Rows after cleaning:", len(df))
print("Duplicate texts:", int(df["text"].duplicated().sum()))
print("Dataset labels are valid.")

Rows before cleaning: 9632
Rows after cleaning: 9632
Duplicate texts: 48
Dataset labels are valid.


## 3. Train / validation / test split and vocabulary

In [4]:
# 80% train, 10% validation, 10% test.
# Random splitting is used because this is a multi-label task.
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED)

def tokenize(text):
    # The supplied Burmese examples are already whitespace-separated.
    return str(text).split()

PAD_IDX = 0
UNK_IDX = 1
counter = Counter()
for text in train_df["text"]:
    counter.update(tokenize(text))

vocab = {"<PAD>": PAD_IDX, "<UNK>": UNK_IDX}
for token, count in counter.items():
    if count >= 1:
        vocab[token] = len(vocab)

print("Vocabulary size:", len(vocab))
print("Train:", len(train_df), "Validation:", len(val_df), "Test:", len(test_df))
print("Training label counts:")
display(train_df[LABELS].sum().rename("positive_count").to_frame())

Vocabulary size: 47118
Train: 7705 Validation: 963 Test: 964
Training label counts:


,positive_count
stereotype,1210
vilification,2549
dehumanization,820
extreme_language,1597
lack_of_empathy,913
invalidation,1238


## 4. Dataset and data loaders

In [5]:
class BurmesePolarizationDataset(Dataset):
    def __init__(self, dataframe, vocab):
        self.texts = dataframe["text"].tolist()
        self.labels = dataframe[LABELS].to_numpy(dtype=np.float32)
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = tokenize(self.texts[idx])
        token_ids = [self.vocab.get(token, UNK_IDX) for token in tokens]
        if not token_ids:
            token_ids = [UNK_IDX]
        return (
            torch.tensor(token_ids, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )

def collate_fn(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(seq) for seq in sequences], dtype=torch.long)
    padded = pad_sequence(sequences, batch_first=True, padding_value=PAD_IDX)
    return padded, lengths, torch.stack(labels)

BATCH_SIZE = 32
train_loader = DataLoader(BurmesePolarizationDataset(train_df, vocab), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(BurmesePolarizationDataset(val_df, vocab), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(BurmesePolarizationDataset(test_df, vocab), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

for name, loader in [("Train", train_loader), ("Validation", val_loader), ("Test", test_loader)]:
    for batch_idx, (_, _, labels) in enumerate(loader):
        assert torch.isfinite(labels).all(), f"{name}: invalid labels in batch {batch_idx}"
    print(f"{name}: all labels are finite")

Train: all labels are finite
Validation: all labels are finite
Test: all labels are finite


## 5. BiLSTM model

In [6]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=128,
                 num_classes=len(LABELS), dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, lengths):
        embedded = self.embedding(x)
        packed = pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (hidden, _) = self.lstm(packed)
        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]
        combined = torch.cat((forward_hidden, backward_hidden), dim=1)
        return self.fc(self.dropout(combined))  # logits; sigmoid is applied later

## 6. Initialize model and train

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BiLSTMClassifier(
    vocab_size=len(vocab),
    embedding_dim=128,
    hidden_dim=128,
    num_classes=len(LABELS),
    dropout=0.3
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
print("Device:", device)
print("Number of labels:", len(LABELS))
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

Device: cuda
Number of labels: 6
Trainable parameters: 6296838


In [8]:
def run_epoch(loader, training=False):
    model.train(training)
    total_loss = 0.0

    for batch_idx, (x, lengths, y) in enumerate(loader):
        x, y = x.to(device), y.to(device)
        if not torch.isfinite(y).all():
            raise ValueError(f"Invalid labels in batch {batch_idx}")

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits = model(x, lengths)
            if not torch.isfinite(logits).all():
                raise ValueError(f"NaN/Inf logits in batch {batch_idx}")
            loss = criterion(logits, y)
            if not torch.isfinite(loss):
                raise ValueError(f"NaN/Inf loss in batch {batch_idx}")
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0, error_if_nonfinite=True)
                optimizer.step()

        total_loss += loss.item() * x.size(0)

    return total_loss / len(loader.dataset)

best_val_loss = float("inf")
best_state = None
patience = 5
wait = 0
EPOCHS = 30

for epoch in range(EPOCHS):
    train_loss = run_epoch(train_loader, training=True)
    val_loss = run_epoch(val_loader, training=False)
    print(f"Epoch {epoch + 1:02d} | Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1

    if wait >= patience:
        print("Early stopping")
        break

if best_state is None:
    raise RuntimeError("No valid model checkpoint was created.")
model.load_state_dict(best_state)
print("Best validation loss:", round(best_val_loss, 4))

Epoch 01 | Train loss: 0.6055 | Val loss: 0.5039
Epoch 02 | Train loss: 0.4727 | Val loss: 0.4812
Epoch 03 | Train loss: 0.4546 | Val loss: 0.4696
Epoch 04 | Train loss: 0.4421 | Val loss: 0.4609
Epoch 05 | Train loss: 0.4327 | Val loss: 0.4594
Epoch 06 | Train loss: 0.4245 | Val loss: 0.4594
Epoch 07 | Train loss: 0.4159 | Val loss: 0.4593
Epoch 08 | Train loss: 0.4063 | Val loss: 0.4547
Epoch 09 | Train loss: 0.3975 | Val loss: 0.4567
Epoch 10 | Train loss: 0.3863 | Val loss: 0.4566
Epoch 11 | Train loss: 0.3757 | Val loss: 0.4580
Epoch 12 | Train loss: 0.3651 | Val loss: 0.4644
Epoch 13 | Train loss: 0.3539 | Val loss: 0.4731
Early stopping
Best validation loss: 0.4547


In [13]:
def get_probs(loader):
    model.eval()
    probs, targets = [], []
    with torch.no_grad():
        for x, lengths, y in loader:
            logits = model(x.to(device), lengths)
            probs.append(torch.sigmoid(logits).cpu().numpy())
            targets.append(y.numpy())
    return np.concatenate(probs), np.concatenate(targets)

val_probs, val_targets = get_probs(val_loader)
test_probs, test_targets = get_probs(test_loader)

# Find the best threshold per label on the VALIDATION set
best_thr = []
for i, label in enumerate(LABELS):
    scores = [
        (f1_score(val_targets[:, i], val_probs[:, i] >= t, zero_division=0), t)
        for t in np.arange(0.05, 0.60, 0.01)
    ]
    best_f1, thr = max(scores)
    best_thr.append(thr)
    print(f"{label:15s} threshold={thr:.2f}  val F1={best_f1:.3f}")
print()

best_thr = np.array(best_thr)

# Apply those thresholds to the TEST set
test_preds = (test_probs >= best_thr).astype(int)

print(classification_report(
    test_targets, test_preds,
    target_names=LABELS, zero_division=0
))

stereotype      threshold=0.09  val F1=0.349
vilification    threshold=0.30  val F1=0.581
dehumanization  threshold=0.12  val F1=0.326
extreme_language threshold=0.12  val F1=0.384
lack_of_empathy threshold=0.18  val F1=0.328
invalidation    threshold=0.15  val F1=0.367

                  precision    recall  f1-score   support

      stereotype       0.19      0.80      0.30       142
    vilification       0.48      0.62      0.54       343
  dehumanization       0.16      0.53      0.25       103
extreme_language       0.25      0.77      0.38       212
 lack_of_empathy       0.22      0.46      0.29       111
    invalidation       0.26      0.54      0.35       189

       micro avg       0.26      0.64      0.37      1100
       macro avg       0.26      0.62      0.35      1100
    weighted avg       0.30      0.64      0.39      1100
     samples avg       0.17      0.33      0.21      1100



## 7. Tune thresholds on validation data and evaluate on test data

In [9]:
def get_probs(loader):
    model.eval()
    probs, targets = [], []
    with torch.no_grad():
        for x, lengths, y in loader:
            logits = model(x.to(device), lengths)
            probs.append(torch.sigmoid(logits).cpu().numpy())
            targets.append(y.numpy())
    return np.concatenate(probs), np.concatenate(targets)

val_probs, val_targets = get_probs(val_loader)
test_probs, test_targets = get_probs(test_loader)

# Choose a separate threshold for each label using validation data only.
best_thr = []
for i, label in enumerate(LABELS):
    scores = [
        (f1_score(val_targets[:, i], val_probs[:, i] >= t, zero_division=0), t)
        for t in np.arange(0.05, 0.96, 0.01)
    ]
    best_f1, thr = max(scores, key=lambda item: item[0])
    best_thr.append(thr)
    print(f"{label:20s} threshold={thr:.2f} | validation F1={best_f1:.3f}")

best_thr = np.array(best_thr)
test_preds = (test_probs >= best_thr).astype(int)

print("\nPer-label classification report:")
for i, label in enumerate(LABELS):
    print(f"\n--- {label} ---")
    print(classification_report(
        test_targets[:, i], test_preds[:, i],
        labels=[0, 1], target_names=["negative", "positive"], zero_division=0
    ))

print("\nOverall micro-averaged metrics:")
print({
    "precision": precision_score(test_targets, test_preds, average="micro", zero_division=0),
    "recall": recall_score(test_targets, test_preds, average="micro", zero_division=0),
    "f1": f1_score(test_targets, test_preds, average="micro", zero_division=0),
    "support": int(test_targets.sum()),
})
print("\nOverall macro F1:", f1_score(test_targets, test_preds, average="macro", zero_division=0))

stereotype           threshold=0.09 | validation F1=0.349
vilification         threshold=0.30 | validation F1=0.581
dehumanization       threshold=0.12 | validation F1=0.326
extreme_language     threshold=0.12 | validation F1=0.384
lack_of_empathy      threshold=0.18 | validation F1=0.328
invalidation         threshold=0.15 | validation F1=0.367

Per-label classification report:

--- stereotype ---
              precision    recall  f1-score   support

    negative       0.92      0.40      0.56       822
    positive       0.19      0.80      0.30       142

    accuracy                           0.46       964
   macro avg       0.55      0.60      0.43       964
weighted avg       0.81      0.46      0.52       964


--- vilification ---
              precision    recall  f1-score   support

    negative       0.75      0.63      0.69       621
    positive       0.48      0.62      0.54       343

    accuracy                           0.63       964
   macro avg       0.62      0.

## 8. Inspect label predictions

In [10]:
print("Label analysis")
print("-" * 90)
for i, label in enumerate(LABELS):
    print(
        f"{label:20s} | Actual positives: {int(test_targets[:, i].sum()):4d} | "
        f"Predicted positives: {int(test_preds[:, i].sum()):4d} | "
        f"Mean probability: {test_probs[:, i].mean():.4f} | "
        f"Threshold: {best_thr[i]:.2f}"
    )

Label analysis
------------------------------------------------------------------------------------------
stereotype           | Actual positives:  142 | Predicted positives:  603 | Mean probability: 0.1519 | Threshold: 0.09
vilification         | Actual positives:  343 | Predicted positives:  444 | Mean probability: 0.3105 | Threshold: 0.30
dehumanization       | Actual positives:  103 | Predicted positives:  345 | Mean probability: 0.1197 | Threshold: 0.12
extreme_language     | Actual positives:  212 | Predicted positives:  652 | Mean probability: 0.1970 | Threshold: 0.12
lack_of_empathy      | Actual positives:  111 | Predicted positives:  236 | Mean probability: 0.1247 | Threshold: 0.18
invalidation         | Actual positives:  189 | Predicted positives:  396 | Mean probability: 0.1644 | Threshold: 0.15


## 9. Save model and preprocessing artifacts

Save the model weights together with the vocabulary, label order, and tuned thresholds. These are required to reproduce predictions later.

In [11]:
import json
from pathlib import Path

OUTPUT_DIR = Path("../models/burmese_polarization_bilstm")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "vocab": vocab,
    "labels": LABELS,
    "embedding_dim": 128,
    "hidden_dim": 128,
    "dropout": 0.3,
    "thresholds": best_thr.tolist(),
    "pad_idx": PAD_IDX,
    "unk_idx": UNK_IDX,
}, OUTPUT_DIR / "model_checkpoint.pt")

with open(OUTPUT_DIR / "vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

with open(OUTPUT_DIR / "labels_and_thresholds.json", "w", encoding="utf-8") as f:
    json.dump({"labels": LABELS, "thresholds": best_thr.tolist()}, f, ensure_ascii=False, indent=2)

print("Saved artifacts to:", OUTPUT_DIR.resolve())

Saved artifacts to: C:\Users\onest\OneDrive\Desktop\Projects\Burmese_Polarization\models\burmese_polarization_bilstm


## Notes

- The six labels are **multi-label**, not mutually exclusive. A text may have multiple labels set to `1`.
- A tiny dataset (such as the 11 example rows in the prompt) is only enough to test that the pipeline runs; it is not enough for a reliable model. Use a substantially larger labeled dataset and inspect per-label support.
- Labels with very few positive examples may be impossible to learn well. Consider collecting more positive examples and checking train/validation/test label counts.
- This notebook keeps the whitespace tokenizer because the sample text appears whitespace-separated. If the source text is not consistently segmented, consider a Burmese syllable/word segmentation strategy and evaluate it against whitespace splitting.
- Thresholds are tuned on the validation split, not the test split, to reduce test-set leakage.